In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

In [ ]:
# Task 1: Write your code here:

csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)



In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
### we have four categorical columns (weather, traffic_level, time_of_day, and vehicle type), need encoding
## we have some missing values as well


In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
## the ranges of the numerical columns are different, some are from 0-9, some are 1-1000, need scaling, I'll probably use min-max since
### it doesnt make since to have negative distance for example

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
### somewhat a bit skewed, might need log transform

In [ ]:
# Task 1: Write your code here:
df = df.drop('Order_ID', axis=1)

df.head()

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
### (weather, traffic_level, time_of_day) are categorical, I can fill them with 'unknown', (i dont wanna drop them cause they might be important)
## delivery time and courier experience years I can fill with mean perhaps
df_clean= df

for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df[col] = df[col].fillna('unknown')

## I wanna the missing values with the mean cause I feel like it's better than mode in this situation
df['Delivery_Time'] = df['Delivery_Time'].fillna(df['Delivery_Time'].mean())
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean())

df.tail()

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:

categorical_features = df.select_dtypes(include=['object']).columns

# one-hot encoding using pd.get_dummies
df = pd.get_dummies(df, columns=categorical_features, drop_first=False)

df = df.astype(float)

df.head()

In [ ]:
# Task 5: Write your code here:
# Apply feature scaling for all features (Use StandardScaler)
## wouldn't this cause data leak? scaling before splitting

from sklearn.preprocessing import StandardScaler


features = df.columns.drop("Delivery_Time")

scaler = StandardScaler()

df[features] = scaler.fit_transform(df[features])

df.head()


In [ ]:
# Task 6: Write your code here:
## the target is delivery_time, which is continuous, it is not labeled, so there is no need to check for class imbalance

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score

X = df.drop("Delivery_Time", axis=1).astype(float)
y = df["Delivery_Time"].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
mae_scores = []

n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

model = RandomForestRegressor(n_estimators=200)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
  model.fit(X_train, y_train)

    # Predict
  y_pred = model.predict(X_test)

    # Calculate metrics
  mae_scores.append(mean_absolute_error(y_test, y_pred))



mae_scores = np.array(mae_scores)



In [ ]:
print(f"5-Fold CV Results:")
print(f"MAE:  {mae_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:

### i wanted to return to this once I finished my other notebooks but I ran of time, I promise I know how to fix this
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here:
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
models = {
  "Ridge Regression": Ridge(alpha=1.0, max_iter=10000),
  "LASSO Regression": Lasso(alpha=1.0,  max_iter=10000),
  "Support Vector Machine": SVR(kernel='rbf'),
  "Decision Tree Regressor": DecisionTreeRegressor(max_depth=10),
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "LightGBM": LGBMRegressor(verbose=-1),
  "CatBoost": CatBoostRegressor(verbose=0)
}

all_results = {}

for name in models:
  all_results[name] = {'mse': [], 'rmse': [], 'r2': []}

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = sklearn_mse(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    mae = mae_scores.append(mean_absolute_error(y_test, y_pred))

    # Store results
    all_results[model_name]["mse"].append(mse)
    all_results[model_name]["rmse"].append(rmse)
    all_results[model_name]["r2"].append(r2)
    all_results[model_name]["mae"].append(mae)

In [ ]:
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MSE:  {np.mean(all_results[model_name]['mse']):.4f}")
  print(f"  RMSE: {np.mean(all_results[model_name]['rmse']):.4f}")
  print(f"  R2:    {np.mean(all_results[model_name]['r2']):.4f}")
  print(f"  MAE:    {np.mean(all_results[model_name]['mae']):.4f}")